# Clustering visuel — Illustrations Bernard Salomon (1557)

**Projet** : Gallica Images — Analyse d'illustrations de l'œuvre d'Ovide  
**Date** : Avril 2026

Ce notebook regroupe les 184 illustrations du livre de Bernard Salomon par similarité visuelle,  
à partir des vecteurs CLIP stockés dans l'API Fouille d'image BnF.

---

## 1. Configuration

In [5]:
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import http.server
import threading
import webbrowser
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from PIL import Image
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

BASE_URL = "https://gallica-search-api-preprod.bnf.lajavaness.com"
ARK_SALOMON = "btv1b2200047r"

print("Configuration chargée.")

Configuration chargée.


---
## 2. Récupération des embeddings CLIP

In [6]:
r = requests.get(f"{BASE_URL}/api/ouvrages/{ARK_SALOMON}/illustrations", timeout=30)
illustrations = r.json()

embeddings = []
valides = []

for illus in illustrations:
    metas = illus.get("metas", {})
    emb = metas.get("content_embedding")
    if emb and len(emb) == 768:
        embeddings.append(emb)
        valides.append(illus)

print(f"Illustrations récupérées : {len(illustrations)}")
print(f"Avec embedding CLIP      : {len(embeddings)}")

# Normalisation des vecteurs
X = normalize(np.array(embeddings))
print(f"Shape de la matrice      : {X.shape}")

AttributeError: 'str' object has no attribute 'get'

---
## 3. Méthode du coude — trouver le bon nombre de clusters

On teste K-Means pour k allant de 2 à 20.  
Le coude dans la courbe indique le nombre de clusters naturellement présent dans les données.

In [ ]:
inerties = []
valeurs_k = range(2, 21)

for k in valeurs_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inerties.append(km.inertia_)
    print(f"  k={k:2d} — inertie={km.inertia_:.1f}")

plt.figure(figsize=(10, 5))
plt.plot(valeurs_k, inerties, marker='o', color='steelblue', linewidth=2)
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertie")
plt.title("Méthode du coude — Bernard Salomon 1557")
plt.xticks(valeurs_k)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
#plt.savefig("coude_kmeans.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n→ Choisir k à l'endroit où la courbe s'aplatit (coude).")

---
## 4. Clustering — choisir k

Modifier `N_CLUSTERS` selon le coude observé ci-dessus.

In [ ]:
# Modifier cette valeur selon le coude observé
N_CLUSTERS = 7

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

print(f"Clusters créés : {N_CLUSTERS}")
print()
for i in range(N_CLUSTERS):
    n = list(labels).count(i)
    barre = "█" * n
    print(f"  Cluster {i:2d} : {n:3d} illustrations  {barre}")

---
## 5. Aperçu visuel — 3 exemples par cluster

In [ ]:
fig, axes = plt.subplots(N_CLUSTERS, 3, figsize=(12, N_CLUSTERS * 3))

# Générer les couleurs dynamiquement
cmap = plt.cm.get_cmap("tab20", N_CLUSTERS)
couleurs = [mcolors.to_hex(cmap(i)) for i in range(N_CLUSTERS)]

for cluster_id in range(N_CLUSTERS):
    indices = [i for i, l in enumerate(labels) if l == cluster_id][:3]

    for col, idx in enumerate(indices):
        ax = axes[cluster_id][col]
        illus = valides[idx]
        metas = illus.get("metas", {})
        link = metas.get("link")
        view = illus.get("view_number", "?")

        if link:
            try:
                resp = requests.get(link, timeout=10)
                img = Image.open(BytesIO(resp.content))
                ax.imshow(img, cmap="gray")
            except:
                ax.text(0.5, 0.5, "Non dispo", ha="center",
                        va="center", transform=ax.transAxes, color="gray")

        ax.set_title(f"Cluster {cluster_id} — p.{view}", fontsize=8,
                     color=couleurs[cluster_id])
        ax.axis("off")

    for col in range(len(indices), 3):
        axes[cluster_id][col].axis("off")

plt.suptitle(f"Clusters (k={N_CLUSTERS}) — Bernard Salomon 1557",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("apercu_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : apercu_clusters.png")

---
## 6. Génération du carrousel interactif


In [ ]:
# Générer les couleurs dynamiquement selon N_CLUSTERS
cmap = plt.cm.get_cmap("tab20", N_CLUSTERS)
couleurs = [mcolors.to_hex(cmap(i)) for i in range(N_CLUSTERS)]

html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <title>Clusters visuels — Bernard Salomon 1557</title>
    <style>
        body {{ font-family: Georgia, serif; background: #f8f5f0; margin: 0; padding: 20px; }}
        h1 {{ text-align: center; color: #2c2c2c; font-size: 1.6em; margin-bottom: 5px; }}
        .subtitle {{ text-align: center; color: #666; font-size: 0.9em; margin-bottom: 30px; }}
        .cluster {{ background: white; border-radius: 8px; padding: 20px;
                   margin-bottom: 30px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); }}
        .cluster-header {{ display: flex; align-items: center; margin-bottom: 15px;
                           border-bottom: 2px solid #eee; padding-bottom: 10px;
                           flex-wrap: wrap; gap: 8px; }}
        .cluster-badge {{ width: 18px; height: 18px; border-radius: 50%; flex-shrink: 0; }}
        .cluster-title {{ font-size: 1.1em; font-weight: bold; color: #2c2c2c; }}
        .cluster-count {{ font-size: 0.85em; color: #888; }}
        .label-input {{ width: 250px; padding: 6px 10px; border: 1px solid #ccc;
                        border-radius: 4px; font-size: 0.9em; font-family: Georgia, serif; }}
        .label-input::placeholder {{ color: #bbb; }}
        .carrousel {{ display: flex; align-items: center; justify-content: center; gap: 24px; }}
        .carrousel-center {{ text-align: center; }}
        .carrousel-center img {{ height: 400px; max-width: 320px; object-fit: contain;
                                 border: 2px solid #ddd; border-radius: 6px;
                                 cursor: pointer; display: block; margin: 0 auto; }}
        .carrousel-center img:hover {{ border-color: #888; }}
        .carrousel-info {{ font-size: 0.85em; color: #555; margin-top: 8px; }}
        .carrousel-counter {{ font-size: 0.8em; color: #aaa; margin-top: 4px; }}
        .btn {{ background: #fff; border: 2px solid #ccc; border-radius: 50%;
                width: 48px; height: 48px; font-size: 1.5em; cursor: pointer;
                display: flex; align-items: center; justify-content: center;
                transition: all 0.2s; flex-shrink: 0; user-select: none; }}
        .btn:hover:not(:disabled) {{ background: #eee; border-color: #888; }}
        .btn:disabled {{ opacity: 0.2; cursor: default; }}
    </style>
</head>
<body>
    <h1>Clusters visuels — Illustrations Bernard Salomon (1557)</h1>
    <p class="subtitle">
        {len(valides)} illustrations regroupées automatiquement par similarité visuelle
        (CLIP + K-Means, k={N_CLUSTERS} clusters)<br>
        Naviguez avec les fl&egrave;ches &mdash; cliquez sur une image pour l'ouvrir dans Gallica
        &mdash; nommez chaque cluster dans le champ texte
    </p>
"""

for cluster_id in range(N_CLUSTERS):
    indices = [i for i, l in enumerate(labels) if l == cluster_id]
    couleur = couleurs[cluster_id]
    nb = len(indices)

    images_list = []
    for idx in indices:
        illus = valides[idx]
        metas = illus.get("metas", {})
        url_img = metas.get("link", "")
        view = illus.get("view_number", 0)
        url_gallica = f"https://gallica.bnf.fr/ark:/12148/{ARK_SALOMON}/f{view}.item"
        images_list.append(f'{{"src":"{url_img}","page":{view},"gallica":"{url_gallica}"}}')

    images_js = "[" + ",".join(images_list) + "]"

    html += f"""
    <div class="cluster">
        <div class="cluster-header">
            <div class="cluster-badge" style="background:{couleur}"></div>
            <span class="cluster-title">Cluster {cluster_id}</span>
            <span class="cluster-count">&nbsp;&mdash; {nb} illustrations</span>
            <input class="label-input" type="text"
                   placeholder="Nommer ce cluster..."
                   id="label_{cluster_id}">
        </div>
        <div class="carrousel">
            <button class="btn" id="prev_{cluster_id}" onclick="C{cluster_id}.go(-1)">&#8592;</button>
            <div class="carrousel-center">
                <a id="link_{cluster_id}" href="#" target="_blank">
                    <img id="img_{cluster_id}" src="" alt="illustration">
                </a>
                <div class="carrousel-info">Page <span id="page_{cluster_id}">—</span></div>
                <div class="carrousel-counter">
                    <span id="cur_{cluster_id}">1</span>&nbsp;/&nbsp;{nb}
                </div>
            </div>
            <button class="btn" id="next_{cluster_id}" onclick="C{cluster_id}.go(1)">&#8594;</button>
        </div>
    </div>

    <script>
    var C{cluster_id} = (function() {{
        var imgs = {images_js};
        var pos  = 0;
        function show(i) {{
            pos = i;
            document.getElementById("img_{cluster_id}").src   = imgs[i].src;
            document.getElementById("link_{cluster_id}").href = imgs[i].gallica;
            document.getElementById("page_{cluster_id}").textContent = imgs[i].page;
            document.getElementById("cur_{cluster_id}").textContent  = i + 1;
            document.getElementById("prev_{cluster_id}").disabled = (i === 0);
            document.getElementById("next_{cluster_id}").disabled = (i === imgs.length - 1);
        }}
        show(0);
        return {{ go: function(dir) {{ var n = pos + dir; if (n >= 0 && n < imgs.length) show(n); }} }};
    }})();
    </script>
"""

html += "\n</body>\n</html>\n"

nom_fichier = "clusters_salomon_carrousel.html"
with open(nom_fichier, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✓ Fichier généré : {nom_fichier}")
print(f"  Taille : {len(html):,} caractères")
print(f"  Clusters : {N_CLUSTERS}")
print(f"  Illustrations : {len(valides)}")

---
## 7. Lancement du serveur local et ouverture dans le navigateur

In [ ]:
PORT = 8081

def lancer_serveur():
    dossier = os.path.dirname(os.path.abspath(nom_fichier))
    os.chdir(dossier)
    handler = http.server.SimpleHTTPRequestHandler
    handler.log_message = lambda *args: None  # silence les logs
    with http.server.HTTPServer(("", PORT), handler) as httpd:
        httpd.serve_forever()

# Lancer le serveur dans un thread daemon
thread = threading.Thread(target=lancer_serveur, daemon=True)
thread.start()

url = f"http://localhost:{PORT}/{nom_fichier}"
print(f"✓ Serveur lancé sur le port {PORT}")
print(f"✓ URL : {url}")
print()
print("→ Ouverture automatique dans le navigateur...")
webbrowser.open(url)